In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
import os


In [2]:

import kagglehub


path = kagglehub.dataset_download('puneet6060/intel-image-classification')
print(path)

Using Colab cache for faster access to the 'intel-image-classification' dataset.
/kaggle/input/intel-image-classification


In [3]:
import cv2
import os
import keras
import tensorflow as tf
from sklearn.model_selection import train_test_split
import random
from sklearn.metrics import classification_report

In [6]:
root = "/kaggle/input/intel-image-classification"

In [7]:
Dataset =os.path.join(root,"seg_train","seg_train")

In [8]:
IMAGE_SIZE = (128 , 128)
BATCH_SIZE =128

In [9]:
names=["mountain","street","buildings","sea","forest","glacier"]

# get class names from data

# Label Encoding

In [10]:
label_map = {name: idx for idx, name in enumerate(names)}
label_map

{'mountain': 0,
 'street': 1,
 'buildings': 2,
 'sea': 3,
 'forest': 4,
 'glacier': 5}

# load images

In [11]:

def loadImages(directory , encoding ):
    images = []
    labels = []
    for name , label in encoding.items():
        path = os.path.join(directory,name)
        if not os.path.exists(path):
            continue
        files = os.listdir(path) # get all images per class
        for imgf in files:
            pathimg = os.path.join(path,imgf)
            image = cv2.imread(pathimg)
            if image is not None:
                image =cv2.resize(image,IMAGE_SIZE) # keep img size const
                images.append(image)
                labels.append(label)
    return images, labels



In [12]:
x,y = loadImages(Dataset,label_map)

# shuffle images for better accuracy

In [13]:
def shuffle(images,labels):
    combined = list(zip(images,labels))
    random.shuffle(combined)
    imagesShuffled,labelsShauffled = zip(*combined)
    return np.array(imagesShuffled),np.array(labelsShauffled)

In [14]:
x,y = shuffle(x,y)

In [15]:
x = np.array(x)
y = np.array(y)
x


array([[[[233, 233, 233],
         [229, 229, 229],
         [228, 228, 228],
         ...,
         [150, 145, 146],
         [153, 148, 149],
         [159, 154, 155]],

        [[225, 225, 225],
         [223, 223, 223],
         [226, 226, 226],
         ...,
         [151, 146, 147],
         [155, 150, 151],
         [163, 158, 159]],

        [[228, 228, 228],
         [227, 227, 227],
         [229, 229, 229],
         ...,
         [151, 146, 147],
         [157, 152, 153],
         [167, 162, 163]],

        ...,

        [[157, 166, 170],
         [147, 156, 160],
         [157, 166, 170],
         ...,
         [179, 186, 186],
         [179, 184, 185],
         [169, 173, 174]],

        [[172, 181, 184],
         [158, 167, 171],
         [139, 148, 152],
         ...,
         [187, 193, 193],
         [178, 183, 182],
         [177, 182, 181]],

        [[153, 162, 166],
         [160, 168, 173],
         [150, 158, 162],
         ...,
         [175, 182, 180],
        

In [16]:
y

array([5, 4, 2, ..., 4, 0, 1])

In [17]:
x = x.astype('float32')#normalization by div on 255
numofclasses = len(names)

In [18]:
numofclasses


6

In [19]:
x

array([[[[233., 233., 233.],
         [229., 229., 229.],
         [228., 228., 228.],
         ...,
         [150., 145., 146.],
         [153., 148., 149.],
         [159., 154., 155.]],

        [[225., 225., 225.],
         [223., 223., 223.],
         [226., 226., 226.],
         ...,
         [151., 146., 147.],
         [155., 150., 151.],
         [163., 158., 159.]],

        [[228., 228., 228.],
         [227., 227., 227.],
         [229., 229., 229.],
         ...,
         [151., 146., 147.],
         [157., 152., 153.],
         [167., 162., 163.]],

        ...,

        [[157., 166., 170.],
         [147., 156., 160.],
         [157., 166., 170.],
         ...,
         [179., 186., 186.],
         [179., 184., 185.],
         [169., 173., 174.]],

        [[172., 181., 184.],
         [158., 167., 171.],
         [139., 148., 152.],
         ...,
         [187., 193., 193.],
         [178., 183., 182.],
         [177., 182., 181.]],

        [[153., 162., 166.],
       

Preping the test data

In [20]:
t_data = os.path.join(root,"seg_test","seg_test")
x_test,y_test = loadImages(t_data,label_map)
x_test,y_test = shuffle(x_test,y_test)
x_test =np.array(x_test)
y_test = np.array(y_test)
x_test = x_test.astype('float32')

using the preprocess_input its better for the model

In [21]:
from tensorflow.keras.applications.efficientnet import preprocess_input
x = preprocess_input(x)
x_test = preprocess_input(x_test)

# build model

In [22]:
from tensorflow.keras.applications import EfficientNetB3


In [23]:
base = EfficientNetB3(
    weights='imagenet',
    include_top=False,
    input_shape=(128, 128, 3)
)
base.trainable = False

In [24]:
model = keras.Sequential([
    keras.layers.Input(shape=(128, 128, 3)),
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.15),
    keras.layers.RandomZoom(0.15),
    base,
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(numofclasses, activation="softmax")
])

In [25]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_flip (RandomFlip)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 128, 128, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom (RandomZoom)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb3 (Functional)     │ (None, 4, 4, 1536)     │    10,783,535 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1536)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1536)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       196,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,981,045 (41.89 MB)

 Trainable params: 197,510 (771.52 KB)

 Non-trainable params: 10,783,535 (41.14 MB)

# compile model

In [26]:
model.compile(
    optimizer="Adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# training

In [27]:
Early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    min_delta=0,
    patience=5,
    verbose=0,
    mode="auto",
    restore_best_weights=True,
    start_from_epoch=5,
)

In [ ]:
history = model.fit(
    x,y,
    validation_data=(x_test,y_test),
    batch_size=32,
    epochs=30,

    callbacks=[Early_stopping]
)

Epoch 1/30
439/439 ━━━━━━━━━━━━━━━━━━━━ 789s 2s/step - accuracy: 0.7831 - loss: 0.5669 - val_accuracy: 0.8607 - val_loss: 0.3644
Epoch 2/30
439/439 ━━━━━━━━━━━━━━━━━━━━ 760s 2s/step - accuracy: 0.8261 - loss: 0.4622 - val_accuracy: 0.8740 - val_loss: 0.3299
Epoch 3/30
439/439 ━━━━━━━━━━━━━━━━━━━━ 753s 2s/step - accuracy: 0.8385 - loss: 0.4245 - val_accuracy: 0.8743 - val_loss: 0.3224
Epoch 4/30
439/439 ━━━━━━━━━━━━━━━━━━━━ 731s 2s/step - accuracy: 0.8448 - loss: 0.4146 - val_accuracy: 0.8737 - val_loss: 0.3219
Epoch 5/30
439/439 ━━━━━━━━━━━━━━━━━━━━ 728s 2s/step - accuracy: 0.8480 - loss: 0.4009 - val_accuracy: 0.8793 - val_loss: 0.3095
Epoch 6/30
439/439 ━━━━━━━━━━━━━━━━━━━━ 771s 2s/step - accuracy: 0.8512 - loss: 0.3912 - val_accuracy: 0.8757 - val_loss: 0.3203
Epoch 7/30
439/439 ━━━━━━━━━━━━━━━━━━━━ 750s 2s/step - accuracy: 0.8533 - loss: 0.3899 - val_accuracy: 0.8863 - val_loss: 0.3064
Epoch 8/30
421/439 ━━━━━━━━━━━━━━━━━━━━ 25s 1s/step - accuracy: 0.8582 - loss: 0.3734

# prediction

In [ ]:
y_pred= model.predict(x_test)
y_pred_classes = np.argmax(y_pred,axis=1)
print(classification_report(y,y_pred_classes, target_names=names))